In [70]:
import os 
from typing import Literal
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph, START, END
from langchain_deepseek import ChatDeepSeek
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain.agents import create_agent
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool

In [71]:
from dotenv import load_dotenv
load_dotenv()

True

In [72]:
class MainState(BaseModel):
    question:str=Field(description="user question")
    category:Literal['coding','google_search','study']= Field(default="google_search")
    answer:str=Field(default="")
    
    

In [73]:
class QuestionCategory(BaseModel):
    category:Literal['coding','google_search','study']=Field(description="Question Category",default='google_search')

In [74]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0
)

In [75]:
search=GoogleSerperAPIWrapper(serper_api_key="37c565decdbf74c2ac8ab1e1404783160e4e093d")
@tool
def google_search_tool(query: str) -> str:
    """Search Google for recent information, facts, and news."""
    return search.run(query)

tools = [google_search_tool]

google_agent=create_react_agent(
    model=llm,
    tools=tools,
    prompt="You are a agent search any queastion in google"
    
)


C:\Users\ASUS TUF A15\AppData\Local\Temp\ipykernel_2700\1067715924.py:9: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  google_agent=create_react_agent(


In [76]:
def chack_question_category(state:MainState) -> MainState:
    
    st_llm = llm.with_structured_output(QuestionCategory)
    res = st_llm.invoke(f"I want to know the category of my question, question is : {state.question}")
    state.category=res.category
    return state
        

In [77]:
def route(state: MainState) -> Literal['coding', 'google_search', 'study']:
    
    return state.category

In [78]:
def google_search_node(state: MainState) -> MainState:
    res=google_agent.invoke({"messages":[
        {"role":"user","content":state.question}
    ]})
    state.answer=res["messages"][-1].content
    return state

In [79]:
study_agent = create_react_agent(
    model=llm,
    tools=[],  # যদি কোনো ক্যালকুলেটর বা উইকিপিডিয়া টুল থাকে দিতে পারেন, না থাকলে খালি লিস্ট []
    prompt="You are an expert academic tutor. Explain concepts clearly with structured points."
)
def study_node(state: MainState) -> MainState:
    res=study_agent.invoke({"messages":[
        {"role":"user","content":state.question}
    ]})
    state.answer=res["messages"][-1].content
    return state

C:\Users\ASUS TUF A15\AppData\Local\Temp\ipykernel_2700\805500358.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  study_agent = create_react_agent(


In [80]:
import sys,io

@tool
def python_interpreter(code: str) -> str:
    """
    Executes Python code and returns the standard output or error message.
    Useful for testing algorithms, debugging, running math calculations, or verifying logic.
    """
    old_stdout = sys.stdout
    redirected_output = io.StringIO()
    sys.stdout = redirected_output
    
    try:
        # কোড এক্সিকিউট করা
        exec(code, {})
        output = redirected_output.getvalue()
        return output if output else "Code executed successfully (no output printed)."
    except Exception as e:
        return f"Execution Error: {str(e)}"
    finally:
        sys.stdout = old_stdout

# টুলের লিস্ট
coding_tools = [python_interpreter]

In [81]:
# কোডিং এজেন্ট
coding_agent = create_react_agent(
    model=llm,
    tools=coding_tools,
    prompt=(
        "You are an expert software engineer and Python specialist. "
        "Write clean, optimized, and well-commented code. "
        "When appropriate, execute the code using the python_interpreter tool to verify its correctness before providing the final answer."
    )
)

C:\Users\ASUS TUF A15\AppData\Local\Temp\ipykernel_2700\1402967481.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  coding_agent = create_react_agent(


In [82]:
def coding_node(state: MainState) -> MainState:
    res=coding_agent.invoke({"messages":[
        {"role":"user","content":state.question}
    ]})
    state.answer=res["messages"][-1].content
    return state

In [83]:
# গ্রাফ ইনিশিয়ালাইজেশন
graph = StateGraph(MainState)

# নোডগুলো রেজিস্টার করা
graph.add_node("check_question_category", chack_question_category)

graph.add_node("study_node", study_node)
graph.add_node("google_search_node", google_search_node)
graph.add_node("coding_node", coding_node)

In [84]:
graph.add_edge(START, "check_question_category")
graph.add_conditional_edges(
    "check_question_category",
    route,
    {
        "coding": "coding_node",
        "google_search": "google_search_node",
        "study": "study_node"
    }
)

# প্রতিটি নোডের কাজ শেষ হলে গ্রাফ সমাপ্ত (END) হবে
graph.add_edge("study_node", END)
graph.add_edge("google_search_node", END)
graph.add_edge("coding_node", END)



In [86]:

app = graph.compile()

# ইনভোক
response = app.invoke({"question": "give me a code by oop python  ?"})
print(response)

{'question': 'give me a code by oop python  ?', 'category': 'coding', 'answer': 'Below is a **self‑contained, object‑oriented Python example** that demonstrates the most common OOP concepts:\n\n* **Encapsulation** – keeping data and the methods that operate on it together.  \n* **Inheritance** – building new classes from existing ones.  \n* **Polymorphism** – using a common interface (`area()` / `perimeter()`) for different shapes.  \n* **Abstraction** – defining an abstract base class that forces subclasses to implement required behaviour.\n\nThe program defines a small hierarchy of geometric shapes (`Shape → Rectangle, Circle, Triangle`) and then shows how you can work with them uniformly.\n\n```python\n"""\nOOP Example – Shape hierarchy\n\nAuthor:  Your Name\nDate:    2026‑08‑26\n"""\n\nfrom __future__ import annotations\nfrom abc import ABC, abstractmethod\nimport math\nfrom typing import List\n\n\nclass Shape(ABC):\n    """\n    Abstract base class for all shapes.\n\n    Every con